In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession.builder.master("local[*]")
    .appName("notebook_teste")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
        "org.scala-lang:scala-library:2.13.12",
    )
    # Configuração do Catálogo Iceberg
    .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties")
    .config("spark.log.level", "ERROR")
    .config("spark.eventLog.enabled", "false")
    # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
    .config("spark.sql.catalog.icebergtb.type", "hadoop")
    .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .getOrCreate()
)

spark

In [37]:
fact_production_df = spark.sql(""" select 
                                        row_number() over (order by workorderid) as fact_productionkey, 
                                        cast(actualcost as decimal(12,2)) as actualcost_ds,
                                        cast(plannedcost as decimal(12,2)) as plannedcost_ds,
                                        *

                                    from icebergtb.advworks_rdb.raw_fact_production""")

In [38]:
fact_production_df = fact_production_df.fillna(
    {
        "scrapreasonid": 0,
        "locationid": 0,
        "actualresourcehrs": 0.00,
        "plannedcost": 0.00,
        "plannedcost_ds": 0.00,
        "actualcost_ds": 0.00,
        "actualcost": 0.00,
        "scheduledstartdate": "2100-12-31 00:00:00",
        "scheduledenddate": "2100-12-31 00:00:00",
        "actualstartdate": "2100-12-31 00:00:00",
        "actualenddate": "2100-12-31 00:00:00",
    }
)

In [41]:
fact_production_df = fact_production_df

fact_production = fact_production_df.select(
    "fact_productionkey",
    "workorderid",
    "productid",
    "orderqty",
    "stockedqty",
    "scrappedqty",
    "scrapreasonid",
    "locationid",
    "actualresourcehrs",
    "plannedcost",
    "plannedcost_ds",
    "actualcost",
    "actualcost_ds",
    "scheduledstartdate",
    "scheduledenddate",
    "actualstartdate",
    "actualenddate",
    "startdate",
    "enddate",
    "duedate",
    "inserted_date",
    "modified_date",
    "eltversion",
)